In [2]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.input.loaders.dfs import (
    store_entity_semantic_embeddings,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.question_gen.local_gen import LocalQuestionGen
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore

## Local Search Example

Local search method generates answers by combining relevant data from the AI-extracted knowledge-graph with text chunks of the raw documents. This method is suitable for questions that require an understanding of specific entities mentioned in the documents (e.g. What are the healing properties of chamomile?).

### Load text units and graph data tables as context for local search

- In this test we first load indexing outputs from parquet files to dataframes, then convert these dataframes into collections of data objects aligning with the knowledge model.

### Load tables to dataframes

In [3]:
# 步骤 1：找到排序最大的文件夹
output_path = "/home/ljc/data/graphrag/alltest/ablation_temp/dataset4_v3_white_t2_multi_single_keep1_enhance_1/output/"
folders = [os.path.join(output_path, d) for d in os.listdir(output_path) if os.path.isdir(os.path.join(output_path, d))]
latest_folder = max(folders, key=os.path.getmtime)

In [4]:
INPUT_DIR = latest_folder + "/artifacts"
LANCEDB_URI = f"{INPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "create_final_community_reports"
ENTITY_TABLE = "create_final_nodes"
ENTITY_EMBEDDING_TABLE = "create_final_entities"
RELATIONSHIP_TABLE = "create_final_relationships"
COVARIATE_TABLE = "create_final_covariates"
TEXT_UNIT_TABLE = "create_final_text_units"
COMMUNITY_LEVEL = 2

#### Read entities

In [5]:
# read nodes table to get community and degree data
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)

# load description embeddings to an in-memory lancedb vectorstore
# to connect to a remote db, specify url and port values.
description_embedding_store = LanceDBVectorStore(
    collection_name="entity_description_embeddings",
)
description_embedding_store.connect(db_uri=LANCEDB_URI)
entity_description_embeddings = store_entity_semantic_embeddings(
    entities=entities, vectorstore=description_embedding_store
)

print(f"Entity count: {len(entity_df)}")
entity_df.head()

Entity count: 1332


,level,title,type,description,source_id,community,degree,human_readable_id,id,size,graph_embedding,entity_type,top_level_node_id,x,y
0,0,BEIJING,GEO,"Beijing is the capital city of China, located ...","32a49f8a4140eaabc660e71ce74a8814,593fb485c8a31...",4,13,0,b45241d70f0e43fca764df95b2b81f77,13.0,"[-0.10995566099882126, -0.03172741457819939, 0...",None,b45241d70f0e43fca764df95b2b81f77,23.767221,5.881624
1,0,CHINA,GEO,"China is a country located in East Asia, known...","174bfa82c2f82d7ab1c9f7942e36080c,593fb485c8a31...",8,11,1,4119fd06010c494caa07f439b333f4c5,11.0,"[-0.040678806602954865, 0.028996486216783524, ...",None,4119fd06010c494caa07f439b333f4c5,-6.275237,20.841255
2,0,BYTEDANCE LTD.,ORGANIZATION,ByteDance Ltd. is a Chinese internet technolog...,593fb485c8a31a331ed213896989c1e0,4,3,2,d3835bf3dda84ead99deadbeac5d0d7d,3.0,"[-0.04495304450392723, -0.06592300534248352, 0...",None,d3835bf3dda84ead99deadbeac5d0d7d,23.078218,7.382460
3,0,ZHANG YIMING,PERSON,Zhang Yiming is a co-founder of ByteDance Ltd....,593fb485c8a31a331ed213896989c1e0,4,1,3,077d2820ae1845bcbb1803379a3d1eae,1.0,"[-0.03439294546842575, -0.04769797995686531, 0...",None,077d2820ae1845bcbb1803379a3d1eae,22.773144,7.544906
4,0,LIANG RUBO,PERSON,"Liang Rubo is a co-founder of ByteDance Ltd., ...",593fb485c8a31a331ed213896989c1e0,4,1,4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,1.0,"[-0.03732438385486603, -0.0586717426776886, 0....",None,3671ea0dd4e84c1a9b02c5ab2c8f4bac,22.959789,7.736664


In [6]:
entity_embedding_df

,id,name,type,description,human_readable_id,graph_embedding,text_unit_ids,description_embedding
0,b45241d70f0e43fca764df95b2b81f77,BEIJING,GEO,"Beijing is the capital city of China, located ...",0,"[-0.10995566099882126, -0.03172741457819939, 0...","[32a49f8a4140eaabc660e71ce74a8814, 593fb485c8a...","[0.046441808342933655, 0.01367973443120718, 0...."
1,4119fd06010c494caa07f439b333f4c5,CHINA,GEO,"China is a country located in East Asia, known...",1,"[-0.040678806602954865, 0.028996486216783524, ...","[174bfa82c2f82d7ab1c9f7942e36080c, 593fb485c8a...","[0.024835562333464622, -0.020249664783477783, ..."
2,d3835bf3dda84ead99deadbeac5d0d7d,BYTEDANCE LTD.,ORGANIZATION,ByteDance Ltd. is a Chinese internet technolog...,2,"[-0.04495304450392723, -0.06592300534248352, 0...",[593fb485c8a31a331ed213896989c1e0],"[0.02458794414997101, -0.04983982816338539, 0...."
3,077d2820ae1845bcbb1803379a3d1eae,ZHANG YIMING,PERSON,Zhang Yiming is a co-founder of ByteDance Ltd....,3,"[-0.03439294546842575, -0.04769797995686531, 0...",[593fb485c8a31a331ed213896989c1e0],"[0.049746934324502945, -0.01967865601181984, -..."
4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,LIANG RUBO,PERSON,"Liang Rubo is a co-founder of ByteDance Ltd., ...",4,"[-0.03732438385486603, -0.0586717426776886, 0....",[593fb485c8a31a331ed213896989c1e0],"[0.04456207901239395, -0.02568388357758522, -0..."
...,...,...,...,...,...,...,...,...
161,83c76fbd2a004d90a5b0a6736ffed61d,NOTRE-DAME DE PARIS,ORGANIZATION,Notre-Dame de Paris is a medieval Catholic cat...,328,"[-0.030096840113401413, 0.028689932078123093, ...",[e8609c1d97f27f704c81b1ae0dfe6a80],"[0.031648579984903336, -0.008553420193493366, ..."
162,d9779c41e3c74fe0b26e23822a4b995b,ÎLE-DE-FRANCE,GEO,Île-de-France is the region that encompasses P...,329,"[-0.026269664987921715, 0.015309534966945648, ...",[e8609c1d97f27f704c81b1ae0dfe6a80],"[0.04107361286878586, -0.012045249342918396, 0..."
163,9d7a563b3b2d405092c31f1fe08cff77,GUSTAVE EIFFEL,PERSON,Gustave Eiffel was the engineer who designed a...,330,"[-0.049480654299259186, 0.043452076613903046, ...",[e8609c1d97f27f704c81b1ae0dfe6a80],"[0.03404342755675316, -0.025873005390167236, -..."
164,bd43f3d439a54781bd4b721a9a269b92,PHILIP II,PERSON,Philip II was the king under whom the Louvre P...,331,"[-0.04972098022699356, 0.04672703146934509, -0...",[e8609c1d97f27f704c81b1ae0dfe6a80],"[-0.008207927457988262, -0.02046208642423153, ..."


In [7]:
entity_embedding_df.to_csv("/home/ljc/data/graphrag/alltest/ablation_temp/dataset4_v3_white_t2_multi_single_keep1_enhance_1/entity.csv",index = False)

#### Read relationships

In [8]:
relationship_df = pd.read_parquet(f"{INPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 443


,source,target,weight,description,text_unit_ids,id,human_readable_id,source_degree,target_degree,rank
0,BEIJING,CHINA,19.0,"Beijing is the capital city of China, making i...","[593fb485c8a31a331ed213896989c1e0, 8f498a85024...",225105a7be14447cb03186bd40756059,0,13,11,24
1,BEIJING,BYTEDANCE LTD.,8.0,"ByteDance Ltd. is headquartered in Beijing, in...",[593fb485c8a31a331ed213896989c1e0],efce8a9d61254447a26aee99e53f0398,1,13,3,16
2,BEIJING,BAIDU,8.0,"Baidu is headquartered in Beijing, which is cr...",[593fb485c8a31a331ed213896989c1e0],4a75a9f0b18a48bea9c0601c0fc395c4,2,13,3,16
3,BEIJING,JD.COM,8.0,"JD.com is headquartered in Beijing, indicating...",[593fb485c8a31a331ed213896989c1e0],e19287afe00a431f9a593a4827d1b448,3,13,2,15
4,BEIJING,PEKING UNIVERSITY,9.0,"Peking University is located in Beijing, makin...",[593fb485c8a31a331ed213896989c1e0],f2c06f3a0c704296bf3353b91ee8af47,4,13,1,14


In [9]:
# covariate_df = pd.read_parquet(f"{INPUT_DIR}/{COVARIATE_TABLE}.parquet")

# claims = read_indexer_covariates(covariate_df)

# print(f"Claim records: {len(claims)}")
# covariates = {"claims": claims}

#### Read community reports

In [10]:
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 65


,community,full_content,level,rank,title,rank_explanation,summary,findings,full_content_json,id
0,63,# Norman Conquest and Its Impact on England\n\...,3,8.5,Norman Conquest and Its Impact on England,The impact severity rating is high due to the ...,The community centers around the Norman Conque...,[{'explanation': 'The Norman Conquest is recog...,"{\n ""title"": ""Norman Conquest and Its Impac...",1419bb59-a57a-4708-ad3f-21f65e50dd56
1,64,# England: Historical Influence and Developmen...,3,7.5,England: Historical Influence and Development,The impact severity rating is high due to Engl...,"The community centers around England, its hist...",[{'explanation': 'England is a crucial constit...,"{\n ""title"": ""England: Historical Influence...",01e66496-3184-4765-a878-b6d044e1effa
2,50,# United Kingdom and Its Global Influence\n\nT...,2,8.5,United Kingdom and Its Global Influence,The impact severity rating is high due to the ...,"The community encompasses the United Kingdom, ...",[{'explanation': 'The United Kingdom is recogn...,"{\n ""title"": ""United Kingdom and Its Global...",ea6f5265-0327-4eea-8da2-406514a925bd
3,51,# Ireland and the United Kingdom: Cultural and...,2,7.5,Ireland and the United Kingdom: Cultural and E...,The impact severity rating is high due to the ...,The community centers around the relationship ...,[{'explanation': 'The Good Friday Agreement of...,"{\n ""title"": ""Ireland and the United Kingdo...",41b599be-0e63-414a-8725-d285987b1235
4,52,# Idaho and Its Economic Ties\n\nThe community...,2,7.5,Idaho and Its Economic Ties,The impact severity rating is high due to Idah...,"The community centers around Idaho, a U.S. sta...",[{'explanation': 'Idaho is strategically locat...,"{\n ""title"": ""Idaho and Its Economic Ties"",...",988dc5f7-ad2f-4e58-b8cb-c14a6bfea3c0


In [11]:
report_df.iloc[0,1]

"# Norman Conquest and Its Impact on England\n\nThe community centers around the Norman Conquest, a pivotal event in 1066 that significantly influenced the historical trajectory of England, particularly the Kingdom of Wessex and London. The relationships among these entities highlight the profound changes in governance, culture, and societal structure that emerged from this historical milestone.\n\n## The significance of the Norman Conquest\n\nThe Norman Conquest is recognized as a pivotal event in English history, marking a transformative period that reshaped the governance and culture of England. This event established the foundation for the modern English state, influencing various aspects of society, including law, language, and architecture. The conquest's implications extended beyond England, affecting the broader historical landscape of the United Kingdom and even France. [Data: Entities (207, 280); Relationships (137, 286)]\n\n## Impact on the Kingdom of Wessex\n\nThe Kingdom o

#### Read text units

In [12]:
text_unit_df = pd.read_parquet(f"{INPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

Text unit records: 53


,id,text,n_tokens,document_ids,entity_ids,relationship_ids
0,593fb485c8a31a331ed213896989c1e0,Beijing is the capital of China. With more tha...,973,[1bb17ced13db9380507b7a26b04f5a19],"[b45241d70f0e43fca764df95b2b81f77, 4119fd06010...","[225105a7be14447cb03186bd40756059, efce8a9d612..."
1,499dc5ce770f5d61a8ec481c94cf36da,North America is a continent[b] in the Norther...,639,[35a3ae500d17b0a3f5a24a9a161c96fd],"[e2f5735c7d714423a2c4f61ca2644626, deece7e64b2...","[aefde1f7617f4c0e9aed31db77f6d862, ad52ba79a84..."
2,d87be950b9b98c10680a005a05cf8852,"Washington, D.C., formally the District of Col...",727,[48d0c023c4e4f58f80b623b95de57d53],"[3d6b216c14354332b1bf1927ba168986, 17ed1d92075...","[f58813d090b947a48c1b4614b92c3ec3, 30a251bc3d0..."
3,818425c6c907abb12a718840abc685a8,The United Nations Security Council (UNSC) is ...,1200,[4c171a717a03a8f18f22d93592d9e67d],"[e657b5121ff8456b9a610cfaead8e0cb, 958beecdb5b...","[ce36d1d637cf4a4e93f5e37ffbc6bd76, cde2d75c51d..."
4,295cb16b56dfd5a9d1041d49ef18028d,", known as the City, is one of the world's lea...",1200,[4c171a717a03a8f18f22d93592d9e67d],"[1745a2485a9443bab76587ad650e9be0, 1eb829d0ace...","[8fba1fea719d49d380ac2d9c310d68b3, 532da08f04f..."


### Create local search context builder

In [13]:
api_key = os.getenv('OPENAI_API_KEY')
llm_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-small"

llm = ChatOpenAI(
    api_key=api_key,
    model=llm_model,
    api_type=OpenaiApiType.OpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=None,
    api_type=OpenaiApiType.OpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    max_retries=20,
)

In [14]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    # covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

### Create local search engine

In [15]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities and relationships. Sum of text_unit_prop and community_prop should be <= 1
# conversation_history_max_turns: maximum number of turns to include in the conversation history.
# conversation_history_user_turns_only: if True, only include user queries in the conversation history.
# top_k_mapped_entities: number of related entities to retrieve from the entity description embedding store.
# top_k_relationships: control the number of out-of-network relationships to pull into the context window.
# include_entity_rank: if True, include the entity rank in the entity table in the context window. Default entity rank = node degree.
# include_relationship_weight: if True, include the relationship weight in the context window.
# include_community_rank: if True, include the community rank in the context window.
# return_candidate_context: if True, return a set of dataframes containing all candidate entity/relationship/covariate records that
# could be relevant. Note that not all of these records will be included in the context window. The "in_context" column in these
# dataframes indicates whether the record is included in the context window.
# max_tokens: maximum number of tokens to use for the context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

llm_params = {
    "max_tokens": 2_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.0,
}

In [16]:
search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraph",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)

### Run local search on sample queries

In [43]:
query = """
What is the capital city of the country that has a historical empire and cultural contributions recognized by France?
"""
result = await search_engine.asearch(query)
print(result.response)

# Capital City of Spain

The capital city of the country that has a historical empire and cultural contributions recognized by France is **Madrid**. Spain, known for its rich history and diverse culture, has played a significant role in shaping European and global affairs. The historical relationship between Spain and France is characterized by cultural exchanges and diplomatic relations, which have influenced both nations over the centuries [Data: Relationships (303, 304)].

## Spain's Historical Context

Spain's historical empire was one of the largest in history, particularly during the Age of Exploration when it established colonies across the Americas, Asia, and Africa. This expansive empire contributed to Spain's cultural richness, which includes art, architecture, and literature that have been recognized and appreciated by France and the broader world [Data: Entities (159)].

## Cultural Contributions

The cultural contributions of Spain are evident in various fields, including 

In [44]:
result.context_data

{'reports':    id                             title  \
 0  44  Eiffel Tower and Parisian Legacy   
 1  44  Eiffel Tower and Parisian Legacy   
 
                                              content  
 0  # Eiffel Tower and Parisian Legacy\n\nThe comm...  
 1  # Eiffel Tower and Parisian Legacy\n\nThe comm...  ,
 'relationships':      id                           source                      target  \
 0   293                            PARIS                      FRANCE   
 1   303                           MADRID                      FRANCE   
 2   304                            SPAIN                      FRANCE   
 3   237                     EIFFEL TOWER                       PARIS   
 4   235               NAPOLEON BONAPARTE                       PARIS   
 5   234               NAPOLEON BONAPARTE                EIFFEL TOWER   
 6   302                           MADRID                       SPAIN   
 7   151                   UNITED KINGDOM                      FRANCE   
 8   147    

In [45]:
result.context_text

'id|title|content\n44|Eiffel Tower and Parisian Legacy|"# Eiffel Tower and Parisian Legacy\n\nThe community centers around the Eiffel Tower, a cultural landmark and symbol of Paris, and its historical connections to Napoleon Bonaparte and urban development plans. The relationships among these entities highlight the Eiffel Tower\'s significance in Paris\'s identity and its evolving role in the city\'s cultural landscape.\n\n## Eiffel Tower as a cultural symbol\n\nThe Eiffel Tower is not only a prominent landmark in Paris but also a global icon representing French culture and heritage. Originally constructed for the 1889 World\'s Fair, it has transcended its initial purpose to become a symbol of Paris, attracting millions of visitors each year. Its designation as the new capital of Paris reflects its dual role as both a tourist attraction and a central figure in the city\'s evolving identity. This significance is underscored by its historical context and the ongoing urban development pla

#### Inspecting the context data used to generate the response

In [46]:
a = result.context_data["entities"]

In [47]:
a

,id,entity,description,number of relationships,in_context
0,151,PARIS,Paris is the capital and largest city of Franc...,9,True
1,97,EIFFEL TOWER,The Eiffel Tower is a significant cultural lan...,5,True
2,223,FRANCE,"France, officially known as the French Republi...",15,True
3,77,EDINBURGH,"Edinburgh is the capital city of Scotland, rec...",7,True
4,96,NAPOLEON BONAPARTE,Napoleon Bonaparte was a prominent military an...,2,True
5,158,MADRID,Madrid is the capital and largest city of Spai...,2,True
6,159,SPAIN,Spain is a country located in Southwestern Eur...,2,True
7,139,ENGLAND,England is a country that is part of the Unite...,15,True
8,65,LONDON,London is a major city that has historically s...,38,True
9,134,CARDIFF,"Cardiff is the capital city of Wales, recogniz...",2,True


In [48]:
df3 = result.context_data["relationships"]

In [49]:
df3

,id,source,target,description,weight,rank,links,in_context
0,293,PARIS,FRANCE,Paris is the capital city of France and serves...,9.0,24,1,True
1,303,MADRID,FRANCE,Madrid's historical relationship with France i...,6.0,17,1,True
2,304,SPAIN,FRANCE,"France shares a maritime border with Spain, in...",7.0,17,2,True
3,237,EIFFEL TOWER,PARIS,"The Eiffel Tower, a prominent landmark in Pari...",17.0,14,1,True
4,235,NAPOLEON BONAPARTE,PARIS,Napoleon Bonaparte significantly influenced th...,9.0,11,2,True
5,234,NAPOLEON BONAPARTE,EIFFEL TOWER,Napoleon Bonaparte played a significant role i...,17.0,7,2,True
6,302,MADRID,SPAIN,"Madrid is the capital city of Spain, serving a...",27.0,4,2,True
7,151,UNITED KINGDOM,FRANCE,France and the United Kingdom are neighboring ...,10.0,57,2,True
8,147,UNITED KINGDOM,EDINBURGH,Edinburgh is the capital city of the United Ki...,18.0,49,2,True
9,193,LONDON,PARIS,London and Paris have historical and cultural ...,5.0,47,2,True


In [50]:
tokyo_university_df = df3[
    (df3["source"].isin(["TOKYO UNIVERSITY", "TOKYO"])) | 
    (df3["target"].isin(["TOKYO UNIVERSITY", "TOKYO"]))
]
tokyo_university_df

,id,source,target,description,weight,rank,links,in_context


In [51]:
result.context_data["reports"]

,id,title,content
0,44,Eiffel Tower and Parisian Legacy,# Eiffel Tower and Parisian Legacy\n\nThe comm...
1,44,Eiffel Tower and Parisian Legacy,# Eiffel Tower and Parisian Legacy\n\nThe comm...


In [52]:
result.context_data["sources"]

,id,text
0,52,Paris (French pronunciation: [paʁi] ⓘ) is the ...
1,45,"center of England, London has been at the for..."
2,44,"the capital city of England, is renowned for ..."
3,19,"France,[a] officially the French Republic,[b] ..."
4,41,state-of-the-art facilities and resources sup...


In [53]:
# if "claims" in result.context_data:
#     print(result.context_data["claims"].head())

### Question Generation

This function takes a list of user queries and generates the next candidate questions.

In [54]:
# question_generator = LocalQuestionGen(
#     llm=llm,
#     context_builder=context_builder,
#     token_encoder=token_encoder,
#     llm_params=llm_params,
#     context_builder_params=local_context_params,
# )
# question_history = [
#     "Tell me about Agent Mercer",
#     "What happens in Dulce military base?",
# ]
# candidate_questions = await question_generator.agenerate(
#     question_history=question_history, context_data=None, question_count=5
# )
# print(candidate_questions.response)

In [55]:
import networkx as nx
from pyvis.network import Network
import random

# Load the GraphML file
G = nx.read_graphml('/data/yuhui/6/graphrag/alltest/location_dataset/dataset_4_revised/output/20241012-123311/artifacts/merged_graph.graphml')
# Create a Pyvis network
net = Network(notebook=True)

# Convert NetworkX graph to Pyvis network
net.from_nx(G)

# Add colors to nodes
for node in net.nodes:
    node['color'] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

# Save and display the network
net.show('knowledge_graph.html')

ModuleNotFoundError: No module named 'pyvis'